First 30-ish trained models
---------------------------

In [3]:
# Import required libraries
from pathlib import Path
import re
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [4]:
# Grab all test data and leave out files
p = Path('../Models')

patternTest = re.compile(r'^Model_Run(?:[1-9]|10)_TestData\.csv$')
testFiles = sorted(str(f) for f in p.rglob('Model_Run*_TestData.csv') if patternTest.match(f.name))

patternLeaveOut = re.compile(r'^Model_Run(?:[1-9]|10)_LeaveOut_TestData\.csv$')
leaveOutFiles = sorted(str(f) for f in p.rglob('Model_Run*_LeaveOut_TestData.csv') if patternLeaveOut.match(f.name))

# Load in Test Data and Leave Out Test Data

In [5]:
datatest = pd.read_csv(r'../Data/DataTest.csv')

In [6]:
datatest

,Lat,Lon,Alt,Date,H2,O18,KPN_A,KPN_B,KPN_C,KPN_D,KPN_E
0,53.871300,8.705800,5.0,2001-02-15 00:00:00+01:00,-57.000000,-8.15000,0,0,1,0,0
1,48.184278,16.389575,196.0,2002-05-24 00:00:00+02:00,-44.800000,-6.36000,0,0,1,0,0
2,51.116535,17.029468,118.0,2005-03-23 00:00:00+01:00,-69.640949,-9.86775,0,0,0,1,0
3,-40.350000,-9.880000,54.0,1995-06-15 00:00:00+02:00,-16.400000,-4.17000,0,0,1,0,0
4,-65.079444,-63.977500,20.0,1987-01-15 00:00:00+01:00,-123.300000,-14.94000,0,0,0,0,1
...,...,...,...,...,...,...,...,...,...,...,...
18336,36.880000,30.700000,49.0,1964-12-15 00:00:00+01:00,-31.200000,-5.35000,0,0,1,0,0
18337,47.470511,21.490419,110.0,2004-08-13 00:00:00+02:00,-24.350000,-4.06000,0,0,0,1,0
18338,47.077778,15.448889,366.0,1990-10-15 00:00:00+02:00,-60.000000,-8.96000,0,0,0,1,0
18339,43.491116,-3.800556,52.0,2009-02-15 00:00:00+01:00,-35.670000,-6.38000,0,0,1,0,0


In [32]:
# Load the data frames into a single DataFrame with a column indicating model type and columns for each run
def load_data(files, path2ogData):
    # Load the base DataFrame from the original data and drop any predicted-value cols
    df = pd.read_csv(path2ogData)
    df = df.drop(columns=['O18 P', 'H2 P'], errors='ignore')

    # Load the original data for the Date column and ensure it's present/parsed
    ogData = pd.read_csv(path2ogData)
    df['Date'] = pd.to_datetime(ogData['Date'], utc=True)

    # Build idCols after ensuring 'Date' exists to avoid duplicates in the id_vars list
    idCols = df.columns.tolist()
    if 'Date' not in idCols:
        idCols.append('Date')

    # Iterate through files and load them into a list of DataFrames
    for file in files:
        # Determine model type and run number from filename
        modelType = file.split('\\')[2]
        modelRun = file.split('\\')[3].split('_')[1]

        # Load the dataframe
        tempDF = pd.read_csv(file)

        # Grab only the predicted columns
        tempDF = tempDF[['O18 P', 'H2 P']]

        # Add those columns to the main DataFrame with appropriate names
        df[f'O18 P-{modelType}-{modelRun}'] = tempDF['O18 P']
        df[f'H2 P-{modelType}-{modelRun}'] = tempDF['H2 P']
    
    # Melt the DataFrame to have a long format
    meltedDF =pd.melt(df, id_vars=idCols, var_name='Model', value_name='Predicted_Value')

    if 'Label' in meltedDF.columns:
        cols = ['Date', 'Lat', 'Lon', 'Alt','Label', 'O18 A', 'H2 A', 'Model', 'Predicted_Value']
    else:
        cols = ['Date', 'Lat', 'Lon', 'Alt', 'O18 A', 'H2 A', 'Model', 'Predicted_Value']
    return meltedDF[cols]
    

In [33]:
testData = load_data(testFiles, r'../Data/DataTest.csv')
testData[['Isotope', 'Model Type', 'Run Number']] = testData['Model'].str.split('-', expand=True)
testData

KeyError: "['O18 A', 'H2 A'] not in index"

In [26]:
pd.read_csv(r'../Data/Leave_Out_Points/Leave_Out_Points_GNIP (2025-07-22).csv')

,Lat,Lon,Alt,Date,H2 A,O18 A,geometry,Label
0,52.3,104.283333,469,2011-06-15T00:00:00.0000000+02:00,-111.2,-14.0,POINT (104.2833333 52.3),lowData
1,52.3,104.283333,469,2011-07-15T00:00:00.0000000+02:00,-74.1,-9.7,POINT (104.2833333 52.3),lowData
2,52.3,104.283333,469,2011-08-15T00:00:00.0000000+02:00,-101.4,-14.2,POINT (104.2833333 52.3),lowData
3,52.3,104.283333,469,2011-09-15T00:00:00.0000000+02:00,-108.0,-13.1,POINT (104.2833333 52.3),lowData
4,52.3,104.283333,469,2011-10-15T00:00:00.0000000+02:00,-127.0,-17.3,POINT (104.2833333 52.3),lowData
...,...,...,...,...,...,...,...,...
1203,18.9,72.820000,10,1961-06-15T00:00:00.0000000+02:00,-10.0,-1.5,POINT (72.82 18.9),hotWet
1204,18.9,72.820000,10,1961-07-15T00:00:00.0000000+02:00,-11.8,-2.1,POINT (72.82 18.9),hotWet
1205,18.9,72.820000,10,1961-08-15T00:00:00.0000000+02:00,3.7,-0.2,POINT (72.82 18.9),hotWet
1206,18.9,72.820000,10,1961-09-15T00:00:00.0000000+02:00,-3.1,-1.2,POINT (72.82 18.9),hotWet


In [27]:
leaveOut = load_data(leaveOutFiles, r'../Data/Leave_Out_Points/Leave_Out_Points_GNIP (2025-07-22).csv')
leaveOut

,Date,Lat,Lon,Alt,O18 A,H2 A,Model,Predicted_Value
0,2011-06-14 22:00:00+00:00,52.3,104.283333,469.0,-14.0,-111.2,O18 P-Global_B-Run10,-20.771523
1,2011-07-14 22:00:00+00:00,52.3,104.283333,469.0,-9.7,-74.1,O18 P-Global_B-Run10,-20.169132
2,2011-08-14 22:00:00+00:00,52.3,104.283333,469.0,-14.2,-101.4,O18 P-Global_B-Run10,-19.360760
3,2011-09-14 22:00:00+00:00,52.3,104.283333,469.0,-13.1,-108.0,O18 P-Global_B-Run10,-18.750465
4,2011-10-14 22:00:00+00:00,52.3,104.283333,469.0,-17.3,-127.0,O18 P-Global_B-Run10,-18.719921
...,...,...,...,...,...,...,...,...
70059,1961-06-14 22:00:00+00:00,18.9,72.820000,10.0,-1.5,-10.0,H2 P-PreWinds_B-Run9,3.623320
70060,1961-07-14 22:00:00+00:00,18.9,72.820000,10.0,-2.1,-11.8,H2 P-PreWinds_B-Run9,-1.493963
70061,1961-08-14 22:00:00+00:00,18.9,72.820000,10.0,-0.2,3.7,H2 P-PreWinds_B-Run9,-3.093568
70062,1961-09-14 22:00:00+00:00,18.9,72.820000,10.0,-1.2,-3.1,H2 P-PreWinds_B-Run9,-5.147001
